# Práctica 4: Dask

Es esta práctica vamos a trabajar con un Dask. Usaremos un dataset complejo con el propósito de evidenciar algunos problemas que pueden surgir en una sesión de trabajo típica.

Ten en cuenta que podrías estar trabajando con dataframes que superan la memoria disponible, lo cual puede tener varias implicaciones en la configuración y el tiempo de procesamiento.

El dataset que vamos a emplear lo puedes descargar de Kaggle **"NYC parking tickets for 2014 to 2017"** proporcionado en https://www.kaggle.com/new-york-city/nyc-parking-tickets


# Entregable 1: 'nombre_apellido_p4_dask.ipynb' 

## 1. Instalación de Dask en local. 

Se recomienda que realices en práctica en Kaggle para tener acceso a varios núcleos de cómputo y aceleración por GPUs. Pero también puedes hacer pruebas en local si lo deseas y tienes los recursos necesarios. Para ello, vamos a crear el nuevo entorno de forma similar a como hemos hecho en prácticas anteriores. Dentro del repo ppd-public, tras hacer pull, veras una nueva carpeta p4 donde crearemos el .env. Los pasos a seguir son los siguientes:
1. Hacemos pull del repo: 
```bash
git pull origin main
```
2. Desde la carpeta environment/p4 creamos el nuevo .env
```bash
uv lock
uv sync --locked
```
3. Ya puedes crear los diferentes documentos zzz.ipynb que se indican el documento del entregable en la carpeta p4. Recuerda seleccionar el entorno correspondiente con Ctrl+Shift+P
  
4. Antes de empezar con el desarrollo de la práctica, tienes que realizar las siguientes acciones:
 
- Comprueba el número de núcleos de CPU en tu configuración (e.g., Kaggle).
- Crea un LocalCluster con una configuración adecuada para los recursos de hardware disponibles.
- Si estás en Colab o Kaggle, utiliza ngrok para exponer el dashboard.

## 2. Cargar y limpiar datos

Una vez configurado el entorno, puedes descargar (o conectar, si estas en Kaggle) el dataset que hemos indicado al principio de este Notebook (https://www.kaggle.com/new-york-city/nyc-parking-tickets) y comenzar con el desarrollo del entregable:

1. Cargar cada archivo CSV en un dataframe de Dask distinto (4 en total) para inspeccionarlos y verificar cualquier problema en cada caso. **Revisa y evidencia los siguientes problemas**:

    - Problema: los tipos de datos inferidos automáticamente por Dask no son correctos.
    - Problema: no todos los archivos tienen las mismas columnas; si simplemente combinamos los 4 datasets, terminaremos con muchos valores nulos.

2. Por lo tanto, antes de combinar los datasets, extraerás solo las columnas comunes entre los 4 archivos.

   Esto se puede hacer de varias maneras, pero una de las más sencillas es utilizar el método `intersection`. Dado que este método funciona con conjuntos de Python, primero convierte la lista de columnas a sets.

```
common_cols = list(set(tickets14.columns).intersection(tickets15.columns, tickets16.columns, tickets17.columns))
```

3. Especifica el esquema de tipo de datos deseado para el DataFrame de Dask: convierte todas las columnas comunes a `'str'`.

4. Ahora que has limpiado los datos, carga los 4 archivos en un único DataFrame de Dask y hazlo persistente en RAM para un procesamiento posterior.

    - Verifica el dashboard y espera hasta que todos los datos estén cargados en RAM. Esto puede tardar varios minutos.
    - Toma una captura de pantalla de tu Dask Dashboard durante el proceso de carga e insértala en tu notebook de entrega. Revisa los bytes almacenados por cada worker y explica el concepto de "Spilling to disk", comentando si te encuentras en esta situación.
    - Verifica y anota el número de particiones del DataFrame.
    - Verifica y anota el número de filas, columnas y el tamaño en RAM del DataFrame.
    


## 3. Calcular algunos filtros

**IMPORTANTE: DESPUES DE CADA APARTADO HAZ UN "del" al dataframe que has creado en el apartado, o eventualmente te quedarás sin espacio en disco para almacenar la información.**

**- Si aún así te sigues quedando sin espacio en disco, (dask usa por defecto /tmp para hacer spill a disco), reduce el número de nodos worker del cluster.**


1. Calcula el condado de NYC que recibió más multas durante el periodo de estudio.
2. Calcula los 10 coches que recibieron más multas durante el periodo de estudio. Utiliza la columna `'Plate ID'` para identificar cada coche.
3. Distribución de multas por mes (agregación temporal): Determinar el número total de multas emitidas en cada mes a lo largo del periodo y descubrir qué mes presenta la mayor cantidad de infracciones (para identificar posibles tendencias estacionales). (Pista: La columna de fecha de emisión Issue Date incluye fecha/hora de la multa): df['Issue Date'].dt.month para obtener el mes).

4. Horas del día con más multas (patrón diario): Analizar en qué horas del día se emiten más multas, identificando las horas pico de sanciones. (Pista: basándote en "Violation Time" extrae la hora del día de cada registro. Crea tu propia funcion "extract_hour" para obtener la hora - el formato es HORAMINUTOA/P, por ejemplo 0710P serían las 19:10. 0810A serían las 8:10-. Utiliza el método "map" para aplicar tu función a la entrada).

5. Tipos de infracción más comunes: Encontrar cuáles son las infracciones de estacionamiento más frecuentes en NYC.

6. Zonas con mayor número de multas: Identificar las ubicaciones de la ciudad con más infracciones registradas (e.g., Calle).

7. Tarea Opcional (4 puntos). Calcula cualquier otro filtro, subconjunto de datos o métrica que consideres relevante en este caso y, al mismo tiempo, que te permita demostrar tus habilidades/experiencia con este tipo de técnicas de preprocesamiento tabular, ¡y obtener mejores calificaciones en tu laboratorio ;-)!
**La actividad opcional solo es necesario que se implemente en una de las plataformas (CPU o GPU).**



# Entregable 2: 'nombre_apellido_p4_dask_category.ipynb'

## 4. Mejorar el uso de memoria

1. Repite todos los pasos de las secciones 1, 2 y 3 anteriores en un nuevo notebook, pero en este caso optimizando el tipo de dato utilizado. Por ejemplo, utiliza el tipo de dato ``'category'`` en lugar de ``'str'`` para las columnas comunes que tengan una cardinalidad baja, y el tipo float para las para las columnas numéricas.

        dtypes = {
            'Summons Number':            'str',   
            'Plate ID':                  'str'
            ...
            'Issuer Code':               'float32',
            'Vehicle Year':              'float32',
            ...
            'Registration State':       'category',
            'Violation County':         'category',
            ...
        }

    NOTA: Es posible que recibas un mensaje de advertencia si utilizas el método ``.groupby`` en el DataFrame (debido al tipo de dato ``'category'``). En ese caso, utiliza el parámetro `observed=True` en el groupby.

2. Vuelve a calcular el tamaño en RAM del DataFrame y aplica los mismos filtros que antes (sección 3). ¿Existen diferencias? Explícalas.



# Entregable 3: ''nombre_apellido_p4_dask_rapids.ipynb'

## 5. RAPIDS: Dask en GPU

1. Repite todos los pasos de las secciones 1, 2 y 3 anteriores en un nuevo notebook, pero en este caso utilizando ``cudf`` o, mejor aún, ``dask_cudf`` para procesar el dataset.

    NOTA: en este caso, utiliza el tipo de dato ``'str'`` para las columnas comunes al leer los archivos en un único ``dask_cudf``.  
    Explicación: El tipo ``category`` de pandas se convierte en CategoricalDtype en la GPU, lo cual no es "hashable" (no se puede usar como clave en un diccionario) y no puede utilizarse durante la reorganización (shuffle) de datos entre particiones.

2. Vuelve a calcular el tamaño en RAM del DataFrame y aplica los mismos filtros que antes. ¿Existen diferencias? Explícalas.



# 6. Conclusiones

Escribe las principales conclusiones obtenidas en este laboratorio, así como cualquier limitación o consideración relacionada con los resultados.
